# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR² dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset metadata is loaded from the Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# (Re-)install mlcroissant if needed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an object, not a dict

print(f"Title: {metadata.name if hasattr(metadata, 'name') else ''}\n\nDescription: {metadata.description if hasattr(metadata, 'description') else ''}")

## 2. Data Overview
List all available record sets and fields, displaying their `@id` and `name` as defined in the Croissant schema. This helps identify how to reference each data block below.


In [ ]:
# Display record sets with their @id and name

record_sets = metadata.record_sets if hasattr(metadata, 'record_sets') else metadata.recordSet
if not record_sets or len(record_sets) == 0:
    print('No record sets found in the metadata. Please check the dataset schema.')
else:
    for rs in record_sets:
        print(f"RecordSet @id: {getattr(rs, '@id', getattr(rs, 'id', ''))}")
        print(f"  Name: {getattr(rs, 'name', '')}")
        if hasattr(rs, 'fields'):
            print('  Fields:')
            for field in rs.fields:
                print(f"    \u2022 Field @id: {getattr(field, '@id', getattr(field, 'id', ''))}  |  Name: {getattr(field, 'name', '')}")
        print('-'*60)

## 3. Data Extraction
Load data from a record set with `mlcroissant`, referencing by `@id` only. Edit the list below to include or exclude record set ids as needed.

In [ ]:
# Find all available record set @ids

# If there are no record sets, skip data extraction
if not record_sets or len(record_sets) == 0:
    dataframes = {}
    print('No record sets available to extract data from.')
else:
    # List all record set @ids
    recset_ids = [getattr(rs, '@id', getattr(rs, 'id', '')) for rs in record_sets]
    print('RecordSet ids:', recset_ids)

    # We'll load the first record set by default
    record_sets_to_load = recset_ids
    dataframes = {}
    for rec_id in record_sets_to_load:
        print(f'Loading records for RecordSet @id: {rec_id}')
        try:
            records = list(dataset.records(record_set=rec_id))
            df = pd.DataFrame(records)
            dataframes[rec_id] = df
            print(f'Loaded {len(df)} records.')
        except Exception as e:
            print(f'Could not load records for {rec_id}:', e)
        print()

    # Show columns for the first loaded record set
    if len(dataframes) > 0:
        sample_recset_id = list(dataframes.keys())[0]
        df = dataframes[sample_recset_id]
        print(f'Columns in record set @id {sample_recset_id}:')
        print(df.columns.tolist())
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data inspection and preprocessing steps, referencing **all fields by their `@id`**. The examples below demonstrate filtering, normalization, and grouping using Croissant `@id` field names.


In [ ]:
if len(dataframes) == 0:
    print('No data to analyze.')
else:
    # Select the first available record set to perform EDA
    eda_recset_id = list(dataframes.keys())[0]
    df = dataframes[eda_recset_id]
    print(f'Running EDA on RecordSet @id: {eda_recset_id}\n')

    # Identify numeric fields (using dtypes)
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    print('Numeric fields (candidate @id):', numeric_cols)
    
    # If none, skip: else pick the first one
    if not numeric_cols:
        print('No numeric field found for analysis.')
    else:
        numeric_field_id = numeric_cols[0]  # Referenced by @id (column name)
        print(f'Analyzing field (by @id): {numeric_field_id}\n')

        # Threshold filtering
        threshold = df[numeric_field_id].median() if not np.isnan(df[numeric_field_id]).all() else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f'Filtered records where {numeric_field_id} > {threshold} (median):')
        display(filtered_df.head())

        # Z-score normalization
        norm_col = f'{numeric_field_id}_normalized'
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (filtered_df[numeric_field_id].std() or 1)
        print(f'Normalized field {numeric_field_id} (new column {norm_col}):')
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a categorical field
        # Choose first non-numeric column
        cat_cols = [c for c in df.columns if c not in numeric_cols]
        if cat_cols:
            group_field_id = cat_cols[0]
            print(f'Grouping filtered data by {group_field_id} (by @id):')
            # Only mean of numeric columns
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            display(grouped_df.head())
        else:
            print('No categorical fields found to group by.')

## 5. Visualization
Visualize the distribution of a numeric field, and relationship with a grouping variable if possible. All columns referenced only by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) == 0:
    print('No data to visualize.')
else:
    # Use the same EDA variables as above
    eda_recset_id = list(dataframes.keys())[0]
    df = dataframes[eda_recset_id]

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    if not numeric_cols:
        print('No numeric columns to visualize.')
    else:
        numeric_field_id = numeric_cols[0]
        plt.figure(figsize=(7,4))
        sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
        plt.title(f'Distribution of {numeric_field_id}')
        plt.xlabel(numeric_field_id)
        plt.ylabel('Count')
        plt.show()

        # Try boxplot by first categorical column
        cat_cols = [c for c in df.columns if c not in numeric_cols]
        if cat_cols:
            group_field_id = cat_cols[0]
            plt.figure(figsize=(10,4))
            sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
            plt.title(f'{numeric_field_id} by {group_field_id}')
            plt.xticks(rotation=45)
            plt.show()

## 6. Conclusion
We demonstrated how to access and analyze the FAIR² mass spectrometry dataset, referencing all data structures and fields strictly by their Croissant `@id`. This reproducible workflow enables transparent and precise data science using standardized metadata, and can be adapted for different Croissant-compliant datasets.

Key findings, outlier samples, normalization steps, and further domain-specific EDA would depend on the schema's rich domain field definition. Remember to adapt record set and field `@id` in the code cells above to your particular analysis or research question.